In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats

# ----------------------------
# Config
ALPHA = 0.05          # left-tail probability (e.g., 0.05 for 95% ES)
N_SIMS = 1_000_000    # number of simulated draws
RANDOM_STATE = 890    # seed for reproducibility
COL_NAME = None       # set a column name string, or leave None to use the first column
# ----------------------------

def expected_shortfall_t_sim(x: np.ndarray,
                             alpha: float = 0.05,
                             n_sims: int = 1_000_000,
                             random_state: int | None = None):
    """
    ES from Simulation using a fitted location-scale Student-t.
    Logic matches the t-distribution task; only change is simulating draws.
    Returns:
        es_abs: positive loss magnitude for ES
        es_diff_from_mean: distance from simulated mean to ES (positive)
        params: (df, mu, sigma)
        var_level: simulated alpha-quantile (left-tail VaR)
        sim_mean: mean of simulated draws
    """
    if not (0 < alpha < 0.5):
        raise ValueError("alpha should be in (0, 0.5).")

    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if x.size == 0:
        raise ValueError("Input series is empty after dropping NaNs.")

    # Fit Student-t on the historical data
    df_t, mu, sigma = stats.t.fit(x)

    # Simulate from fitted t
    rng = np.random.default_rng(random_state)
    sims = stats.t.rvs(df_t, loc=mu, scale=sigma, size=n_sims, random_state=rng)

    # VaR: alpha-quantile (left tail)
    var_level = float(np.quantile(sims, alpha, method="linear"))

    # ES: mean of the tail X | X <= VaR
    tail = sims[sims <= var_level]
    if tail.size == 0:
        raise RuntimeError("No simulated points in the left tail; increase n_sims or check inputs.")
    cond_mean = float(np.mean(tail))     # typically negative for returns
    sim_mean = float(np.mean(sims))

    es_abs = -cond_mean                  # positive loss magnitude
    es_diff_from_mean = sim_mean - cond_mean

    return es_abs, es_diff_from_mean, (df_t, mu, sigma), var_level, sim_mean

def main():
    # Read data
    DATA_DIR = Path.cwd() / "testfiles_" / "data"
    CSV_PATH = DATA_DIR / "test7_2.csv"
    df = pd.read_csv(CSV_PATH, header=0)

    # Select target column
    if COL_NAME is None:
        series = df.iloc[:, 0].values
        used_col = df.columns[0]
    else:
        series = df[COL_NAME].values
        used_col = COL_NAME

    es_abs, es_diff_mean, params, var_level, sim_mean = expected_shortfall_t_sim(
        series, alpha=ALPHA, n_sims=N_SIMS, random_state=RANDOM_STATE
    )
    df_t, mu, sigma = params

    print(f"ES Absolute: {es_abs}")
    print(f"ES Diff from Mean: {es_diff_mean}")

if __name__ == "__main__":
    main()


ES Absolute: 0.0753324053799625
ES Diff from Mean: 0.1213136404971025
